In [2]:
%matplotlib inline

# in case user loads module separately from sapphire.run()
from jax import config as jax_config
jax_config.update("jax_enable_x64", True) # required to accurately solve and take gradients through our diffeqs 
import jax
import jax.numpy as jnp

import numpy as np
from astropy.table import Table, Column
import pandas as pd

In [3]:
# file name of npz with %s for chain num
path_npz = '/mnt/ceph/users/vpandya/sapphire_outputs/outputs/numpyro_mix_chain%s_111_slope4_mzrgas.npz'

In [4]:
###### the following is adapted from sapphire/visualization/numpyro_plots.py 

params_free = ["A_M","alpha0_M","A_E","alpha0_E","A_SF","alpha0_SF","A_Z","alpha0_Z"]

### loop over and store posterior samples from all 4 chains    
# samples dataframes indexed by int for chain_num for Gelman-Rubin statistic 
chain_samples = {}

# TO DO: generalize depending on actual # of chains
for cnum in range(0,4):

    # TO DO: generalize this beyond numpyro_manga_chain%s_%s.npz
    npz_nuts = jnp.load(path_npz%cnum,allow_pickle=True)

    samples_dict = {k: npz_nuts['samples'].item()[k] for k in params_free}
    samples_df = pd.DataFrame(samples_dict)
    chain_samples[cnum] = samples_df

# combine the different chains' posterior samples
samples_df = pd.concat(chain_samples.values(), ignore_index=True)  

samples_df

,A_M,alpha0_M,A_E,alpha0_E,A_SF,alpha0_SF,A_Z,alpha0_Z
0,0.487296,-0.008129,-0.530952,-0.549559,0.881462,-2.675425,-1.659196,-0.563436
1,0.270888,-3.420749,-0.572425,-0.579525,0.871743,-2.661834,-1.258503,-3.172387
2,-0.017579,-1.009934,-0.555337,-1.499298,0.877239,-2.198920,-0.598878,-0.084833
3,0.450659,-0.350230,-0.558869,-0.827851,0.851939,-2.773659,-1.085814,-0.520671
4,0.364610,-1.011424,-0.531380,-0.775790,0.843917,-2.553842,-1.276185,-0.190856
...,...,...,...,...,...,...,...,...
3995,0.079562,-2.843932,-0.498339,-0.321462,0.941044,-1.999538,-1.445528,-2.609273
3996,0.119164,-3.490619,-0.505423,-0.197787,0.938474,-1.983146,-1.419602,-3.215999
3997,0.380484,-1.911299,-0.497261,-0.275667,0.920482,-2.348960,-0.956245,-0.697295
3998,-0.025028,-3.363008,-0.660614,-1.398175,0.810223,-2.588059,-1.972130,-1.123262


In [5]:
### for each parameter, record 16, 50, 84 percentile of posterior

p16, p50, p84 = [], [], []

for pnum,pval in enumerate(params_free):
    
    ptiles = samples_df[pval].quantile([0.16,0.5,0.84])

    p16.append(float(ptiles.iloc[0]))
    p50.append(float(ptiles.iloc[1]))
    p84.append(float(ptiles.iloc[2]))

p16, p50, p84

([-0.01331471780428132,
  -3.312973388654386,
  -0.5896700103148758,
  -1.3248123664289206,
  0.8513458000253655,
  -2.67180030087883,
  -1.768140426541817,
  -2.991594496240657],
 [0.21844597729048315,
  -2.419656534871934,
  -0.5536709873359688,
  -0.9335506030794731,
  0.8838099655553638,
  -2.3423143953300354,
  -1.2143320814997738,
  -1.4409232362861826],
 [0.40351806223476977,
  -1.2792909513659676,
  -0.5146327671813856,
  -0.5880201758378123,
  0.9142721014729513,
  -1.9893872228721274,
  -0.6263471331783793,
  -0.40150742351998875])

In [16]:
### now create an astropy table to save as latex table
parlabels = [r'$A_M$',r'$\alpha_M^0$',r'$A_E$',r'$\alpha_E^0$',
             r'$A_{\rm SF}$',r'$\alpha_{\rm SF}^0$',r'$A_Z$',r'$\alpha_Z^0$']

t1 = Table([Column(parlabels, name=r'$\theta$'),
           Column(p16, name=r'$p_{16}$', format='%.3f'),
           Column(p50, name=r'$p_{50}$', format='%.3f'),
           Column(p84, name=r'$p_{84}$', format='%.3f'),])

t1

$\theta$,$p_{16}$,$p_{50}$,$p_{84}$
str19,float64,float64,float64
$A_M$,-0.013,0.218,0.404
$\alpha_M^0$,-3.313,-2.420,-1.279
$A_E$,-0.590,-0.554,-0.515
$\alpha_E^0$,-1.325,-0.934,-0.588
$A_{\rm SF}$,0.851,0.884,0.914
$\alpha_{\rm SF}^0$,-2.672,-2.342,-1.989
$A_Z$,-1.768,-1.214,-0.626
$\alpha_Z^0$,-2.992,-1.441,-0.402


In [17]:
t1.write('/mnt/ceph/users/vpandya/sapphire_outputs/table1.tex',format='latex',overwrite=True)